# DeepVRegulome — Quickstart Tutorial

This notebook walks you through installing DeepVRegulome, exploring the 462 available models,
and scoring your first variant.

**What you'll learn:**
1. Install the package and verify GPU availability
2. Browse and search 462 fine-tuned DNABERT models
3. Score a single variant with `dvr.score_variant()`
4. Score pre-extracted sequences with `dvr.score_sequence()`
5. Interpret the output columns

**Prerequisites:**
- A reference genome FASTA file (hg38). See the [README](https://github.com/DavuluriLab/DeepVRegulome#external-data-requirements) for download instructions.
- GPU recommended but not required

## 1. Installation

In [1]:
# Install deepvregulome (skip if already installed)
#!pip install deepvregulome[all]

# Upgrade to latest version
#!pip install deepvregulome --upgrade

In [2]:
import pandas as pd
print(pd.__version__)       # should be 2.2.3
print(pd.__file__)          # should show ~/.local/ not /shared/software/

3.0.1
/shared/software/anaconda/anaconda3/envs/dvr/lib/python3.11/site-packages/pandas/__init__.py


In [3]:
# Verify installation
import deepvregulome
print(f"deepvregulome version: {deepvregulome.__version__}")

import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU count: {torch.cuda.device_count()}")

deepvregulome version: 0.1.8
PyTorch version: 2.10.0+cu128
CUDA available: True
GPU: NVIDIA A40
GPU count: 8


---
## 2. Initialize DVR

The `DVR` class is the main interface. Pass the path to your reference genome FASTA.

> **Important:** Do NOT specify `coordinate_system` unless you are certain about your data format.
> DVR automatically detects whether your coordinates are 0-based (BED) or 1-based (VCF)
> by checking your REF alleles against the reference genome.

In [4]:
from deepvregulome import DVR

# ⚠️ Update this path to your local hg38 FASTA file
GENOME = "/data/projects/human_reference_genome/hg38/all_chr_ref.fasta"

dvr = DVR(genome=GENOME)

/shared/software/anaconda/anaconda3/envs/dvr/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


---
## 3. Browse Available Models

DeepVRegulome provides 462 fine-tuned DNABERT models:
- **458 transcription factor (TF)** binding models
- **4 histone modification** models

All trained on ENCODE ChIP-seq data for the human genome (hg38).

In [5]:
# List all models (sorted by ROC-AUC by default)
all_models = dvr.list_models()
print(f"Total models: {len(all_models)}")
print(f"\nTop 10 by ROC-AUC:")
for m in all_models[:10]:
    print(f"  {m.name:<12} {m.model_type:<8} ROC-AUC={m.roc_auc:.2f}  Accuracy={m.accuracy:.1f}%  Peaks={m.peak_count:,}")

Total models: 462

Top 10 by ROC-AUC:
  ZFHX2        TF       ROC-AUC=99.82  Accuracy=94.7%  Peaks=61,052
  NFE2         TF       ROC-AUC=99.76  Accuracy=94.2%  Peaks=53,717
  EGR2         TF       ROC-AUC=99.75  Accuracy=95.7%  Peaks=57,610
  CTCFL        TF       ROC-AUC=99.71  Accuracy=98.4%  Peaks=12,743
  CEBPA        TF       ROC-AUC=99.70  Accuracy=92.4%  Peaks=57,842
  FOS          TF       ROC-AUC=99.62  Accuracy=91.5%  Peaks=169,317
  SCRT2        TF       ROC-AUC=99.56  Accuracy=93.6%  Peaks=59,462
  CEBPB        TF       ROC-AUC=99.54  Accuracy=91.7%  Peaks=148,540
  RUNX3        TF       ROC-AUC=99.50  Accuracy=91.5%  Peaks=69,903
  ZNF143       TF       ROC-AUC=99.48  Accuracy=89.8%  Peaks=66,295


In [6]:
# Filter by model type
tf_models = dvr.list_models(model_type="TF")
histone_models = dvr.list_models(model_type="HISTONE")

print(f"TF models: {len(tf_models)}")
print(f"Histone models: {len(histone_models)}")

print("\nHistone models:")
for m in histone_models:
    print(f"  {m.name:<12} ROC-AUC={m.roc_auc:.2f}  Peaks={m.peak_count:,}")

TF models: 458
Histone models: 4

Histone models:
  H4K12ac      ROC-AUC=96.10  Peaks=42,837
  H2AK9ac      ROC-AUC=94.58  Peaks=129,243
  H3K23me2     ROC-AUC=93.76  Peaks=77,099
  H3K9me1      ROC-AUC=93.75  Peaks=196,657


In [7]:
# Search models by name
print("GATA family:", dvr.search_models("GATA"))
print("ZNF family (first 5):", dvr.search_models("ZNF")[:5], "...")
print("STAT family:", dvr.search_models("STAT"))
print("FOX family:", dvr.search_models("FOX"))

GATA family: ['GATA1', 'GATA2', 'GATA3', 'GATA4', 'GATAD1', 'GATAD2A']
ZNF family (first 5): ['ZNF10', 'ZNF114', 'ZNF121', 'ZNF133', 'ZNF138'] ...
STAT family: ['STAT1', 'STAT2', 'STAT3']
FOX family: ['FOXA1', 'FOXA2', 'FOXA3', 'FOXM1', 'FOXP1', 'FOXP2']


In [8]:
# Get detailed info for a specific model
from deepvregulome import ModelRegistry

info = ModelRegistry().get("CTCFL")
print(f"Model: {info.name}")
print(f"Type: {info.model_type}")
print(f"ROC-AUC: {info.roc_auc}  |  Accuracy: {info.accuracy}%  |  F1: {info.f1}")
print(f"Peaks: {info.peak_count:,}")
print(f"HuggingFace: {info.hf_repo} / {info.subfolder}")

Model: CTCFL
Type: TF
ROC-AUC: 99.71  |  Accuracy: 98.39%  |  F1: 98.4
Peaks: 12,743
HuggingFace: duttaprat/DeepVRegulome / models/CTCFL


---
## 4. Score a Single Variant

`dvr.score_variant()` takes genomic coordinates and returns a DataFrame with:

| Column | Meaning |
|--------|---------|
| `prob_ref` | Binding probability for the wild-type (reference) sequence (0–1) |
| `prob_alt` | Binding probability for the mutant (alternate) sequence (0–1) |
| `log_odds_ratio` | log₂(ref_odds) − log₂(alt_odds). **Positive = loss of binding** |
| `score_change` | (p_alt − p_ref) × max(p_ref, p_alt). Magnitude-aware change score |

The first call triggers a **sanity check** that verifies your REF allele matches the
reference genome and auto-detects the coordinate system.

In [9]:
# Score a single SNV against multiple TF models
result = dvr.score_variant(
    chrom="chr9",
    pos=65385776,
    ref="G",
    alt="A",
    models=["ATF4", "CTCFL", "SP1", "GATA3"],
)

result

⚠ Sanity check: likely 1-based but not fully certain
  Checked 1 variants: 1 suggest 1-based, 0 suggest 0-based, 0 ambiguous, 0 no-match
  Proceeding with 1-based. Override with coordinate_system= if needed.


Loading weights: 100%|█████████████████████████████████████████████████████████████████████████████████████████| 201/201 [00:00<00:00, 33042.84it/s]

Loading weights: 100%|█████████████████████████████████████████████████████████████████████████████████████████| 201/201 [00:00<00:00, 36454.86it/s]

Loading weights: 100%|█████████████████████████████████████████████████████████████████████████████████████████| 201/201 [00:00<00:00, 37065.51it/s]

Loading weights: 100%|█████████████████████████████████████████████████████████████████████████████████████████| 201/201 [00:00<00:00, 34421.65it/s]

Models: 100%|███████████████████████████████████████████████████████████████████████████████████████| 4/4 [00:05<00:00,  1.28s/model, current=GATA3]


,chrom,pos,ref,alt,model,type,prob_ref,prob_alt,log_odds_ratio,score_change
0,chr9,65385776,G,A,ATF4,TF,0.999954,0.102366,17.5287,-0.897546
1,chr9,65385776,G,A,GATA3,TF,0.023859,0.985276,-11.4188,0.947262
2,chr9,65385776,G,A,CTCFL,TF,0.323819,0.870792,-3.8149,0.476300
3,chr9,65385776,G,A,SP1,TF,0.694462,0.843742,-1.2483,0.125954


---
## 5. Score Pre-Extracted Sequences

If you already have 301bp REF and ALT sequences (e.g., from your own pipeline),
use `dvr.score_sequence()` directly. This does not require a reference genome.

In [10]:
# Example: score a pre-extracted 301bp sequence pair
# The variant is at position 151 (center of the 301bp window)
import random
random.seed(42)

# Generate a synthetic 301bp sequence for demonstration
ref_seq = "".join(random.choices("ACGT", k=301))
# Create ALT by mutating the center position
alt_seq = ref_seq[:150] + ("T" if ref_seq[150] != "T" else "A") + ref_seq[151:]

print(f"REF center: ...{ref_seq[145:156]}...")
print(f"ALT center: ...{alt_seq[145:156]}...")
print(f"Mutation at position 151: {ref_seq[150]} → {alt_seq[150]}")

REF center: ...AGGACTGCGAA...
ALT center: ...AGGACAGCGAA...
Mutation at position 151: T → A


In [11]:
# Score without a reference genome — just sequences
dvr_seq = DVR()  # no genome needed for score_sequence

result_seq = dvr_seq.score_sequence(
    ref_seq=ref_seq,
    alt_seq=alt_seq,
    models=["CTCFL", "SP1"],
)
result_seq

Loading weights: 100%|█████████████████████████████████████████████████████████████████████████████████████████| 201/201 [00:00<00:00, 31396.36it/s]

Loading weights: 100%|█████████████████████████████████████████████████████████████████████████████████████████| 201/201 [00:00<00:00, 30172.69it/s]

Models: 100%|█████████████████████████████████████████████████████████████████████████████████████████| 2/2 [00:02<00:00,  1.02s/model, current=SP1]


,model,type,prob_ref,prob_alt,log_odds_ratio,score_change
0,SP1,TF,0.999577,0.999248,0.8315,-0.000329
1,CTCFL,TF,0.000067,0.000068,-0.0325,0.000000


---
## 6. Score Multiple Variants from a DataFrame

`dvr.score_variants()` accepts a pandas DataFrame with columns: `chrom`, `pos`, `ref`, `alt`.

In [12]:
import pandas as pd

# Create a small variant table
variants = pd.DataFrame([
    {"chrom": "chr9",  "pos": 65385776, "ref": "G", "alt": "A"},
    {"chrom": "chr1",  "pos": 3456782,  "ref": "A", "alt": "C"},
    {"chrom": "chr17", "pos": 7674220,  "ref": "C", "alt": "T"},
])

print("Input variants:")
print(variants.to_string(index=False))

Input variants:
chrom      pos ref alt
 chr9 65385776   G   A
 chr1  3456782   A   C
chr17  7674220   C   T


In [13]:
# Score all variants against 2 models
results = dvr.score_variants(
    variants,
    models=["CTCFL", "ATF4"],
    batch_size=32,
)

print(f"\nResults: {len(results)} rows ({len(variants)} variants × 2 models)")
results

Extracting sequences for 3 variants...
Scoring 3 variants × 2 models...


Loading weights: 100%|█████████████████████████████████████████████████████████████████████████████████████████| 201/201 [00:00<00:00, 17947.65it/s]

Loading weights: 100%|█████████████████████████████████████████████████████████████████████████████████████████| 201/201 [00:00<00:00, 14770.49it/s]

Models: 100%|████████████████████████████████████████████████████████████████████████████████████████| 2/2 [00:02<00:00,  1.12s/model, current=ATF4]


Results: 6 rows (3 variants × 2 models)


,chrom,pos,ref,alt,model,type,prob_ref,prob_alt,log_odds_ratio,score_change
0,chr9,65385776,G,A,ATF4,TF,0.999954,0.102366,17.5287,-0.897546
1,chr9,65385776,G,A,CTCFL,TF,0.323814,0.870796,-3.8149,0.476310
2,chr17,7674220,C,T,ATF4,TF,0.000221,0.000325,-0.5561,0.000000
3,chr1,3456782,A,C,CTCFL,TF,0.000329,0.000421,-0.3562,0.000000
4,chr1,3456782,A,C,ATF4,TF,0.000055,0.000056,-0.0426,0.000000
5,chr17,7674220,C,T,CTCFL,TF,0.000088,0.000087,0.0101,-0.000000


---
## 7. GPU and Batch Configuration

For large-scale scoring, adjust batch size and use multiple GPUs:

In [15]:
# Multi-GPU scoring (if available)
results = dvr.score_variants(
    variants,
    models=["CTCFL", "SP1", "ATF4", "GATA3", "MYC"],
    batch_size=100,
    gpus=[0, 1, 2, 3],  # distribute models across GPUs
)

Extracting sequences for 3 variants...
Scoring 3 variants × 5 models...
Distributing 5 models across 4 GPUs: [1, 1, 1, 2] each


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 22589.90it/s]


  [GPU 0] ✓ CTCFL (1/1)
  [GPU 2] ✓ ATF4 (1/1)
  [GPU 1] ✓ SP1 (1/1)
  [GPU 3] ✓ GATA3 (1/2)


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 17654.50it/s]


  [GPU 3] ✓ MYC (2/2)


---
## Summary

| What you want to do | Method | Genome needed? |
|---|---|---|
| Score one variant by coordinates | `dvr.score_variant(chrom, pos, ref, alt, models)` | Yes |
| Score many variants from a DataFrame | `dvr.score_variants(df, models)` | Yes |
| Score from a VCF file | `dvr.score_vcf(path, models)` | Yes |
| Score pre-extracted sequences | `dvr.score_sequence(ref_seq, alt_seq, models)` | No |
| Browse models | `dvr.list_models()`, `dvr.search_models("GATA")` | No |

**Next:** See `02_vcf_scoring.ipynb` for batch VCF analysis and `03_attention_motifs.ipynb` for interpretability.